In [ ]:
# ============================================================
# SIMPLE PORTFOLIO CHATBOT - LSTM
# ============================================================
import re
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from difflib import get_close_matches
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# ============================================================
# 1. LOAD DATASET
# ============================================================
dataset = pd.read_csv("my_portfolio_data.csv")
# Keep only question and answer
dataset = dataset[["question", "answer"]]
# Remove empty rows
dataset = dataset.dropna()
print("Original questions:", len(dataset))
print(dataset.head())

In [ ]:
# ============================================================
# 2. CLEAN TEXT
# ============================================================
def clean_text(text):
    text = str(text).lower().strip()
    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # Remove extra spaces
    text = " ".join(text.split())
    return text

dataset["question"] = dataset["question"].apply(clean_text)
dataset["answer"] = dataset["answer"].apply(lambda x: str(x).strip())

In [ ]:
# ============================================================
# 3. CREATE MORE QUESTION EXAMPLES
# ============================================================

prefixes = ["", "please ", "can you tell me ", "i want to know "]
suffixes = ["", " please", " for me"]
questions = []
answers = []

for row in dataset.itertuples(index=False):
    question = row.question
    answer = row.answer
    for prefix in prefixes:
        for suffix in suffixes:
            new_question = (prefix + question + suffix)
            new_question = clean_text(new_question)
            questions.append(new_question)
            answers.append(answer)

print("Training examples:", len(questions))

In [ ]:
# ============================================================
# 4. ADD START AND END TOKENS
# ============================================================

answers_with_tokens = []

for answer in answers:
    new_answer = ("<start> " + answer + " <end>")
    answers_with_tokens.append(new_answer)

In [ ]:
# ============================================================
# 5. CREATE TOKENIZERS
# ============================================================

question_tokenizer = Tokenizer(filters="", oov_token="<unk>")
answer_tokenizer = Tokenizer(filters="", oov_token="<unk>")

question_tokenizer.fit_on_texts(questions)
answer_tokenizer.fit_on_texts(answers_with_tokens)

In [ ]:
# ============================================================
# 6. CONVERT TEXT INTO NUMBERS
# ============================================================

question_sequences = (question_tokenizer.texts_to_sequences(questions))
answer_sequences = (answer_tokenizer.texts_to_sequences(answers_with_tokens))

In [ ]:
# ============================================================
# 7. FIND MAXIMUM LENGTH
# ============================================================

max_question_length = max(len(x) for x in question_sequences)
max_answer_length = max(len(x) for x in answer_sequences)

print("Maximum question length:", max_question_length)
print("Maximum answer length:", max_answer_length)

In [ ]:
# ============================================================
# 8. ADD PADDING
# ============================================================

question_sequences = pad_sequences(question_sequences, maxlen=max_question_length, padding="post")
answer_sequences = pad_sequences(answer_sequences, maxlen=max_answer_length, padding="post")

In [ ]:
# ============================================================
# 9. CREATE DECODER INPUT AND OUTPUT
# ============================================================

# Example:

# Full:

# <start> I am a developer <end>

# Decoder input:

# <start> I am a developer

# Decoder target:

# I am a developer <end>

decoder_input = answer_sequences[:, :-1]
decoder_target = answer_sequences[:, 1:]

In [ ]:
# ============================================================
# 10. VOCABULARY SIZE
# ============================================================

question_vocab_size = (len(question_tokenizer.word_index) + 1)
answer_vocab_size = (len(answer_tokenizer.word_index) + 1)

print("Question vocabulary:", question_vocab_size)
print("Answer vocabulary:", answer_vocab_size)

In [ ]:
# ============================================================
# 11. CREATE ENCODER
# ============================================================

# Question goes into the encoder
encoder_input = Input(shape=(max_question_length,))
encoder_embedding = Embedding(input_dim=question_vocab_size, output_dim=128, mask_zero=True)(encoder_input)
encoder_lstm = LSTM(128, return_state=True)
encoder_output, state_h, state_c = (encoder_lstm(encoder_embedding))



In [ ]:
# ============================================================
# 12. CREATE DECODER
# ============================================================

decoder_input_layer = Input(shape=(max_answer_length - 1,))
decoder_embedding = Embedding(input_dim=answer_vocab_size, output_dim=128, mask_zero=True)(decoder_input_layer)
decoder_lstm = LSTM(128, return_sequences=True)
decoder_output = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])

In [ ]:
# ============================================================
# 13. OUTPUT LAYER
# ============================================================

output = Dense(answer_vocab_size, activation="softmax")(decoder_output)

In [ ]:
# ============================================================
# 14. CREATE MODEL
# ============================================================

model = Model(inputs=[encoder_input,decoder_input_layer], outputs=output)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# ============================================================
# 15. TRAIN MODEL
# ============================================================

from sklearn.model_selection import train_test_split

X_question_train, X_question_val, X_decoder_train, X_decoder_val, y_train, y_val = train_test_split(
    question_sequences, decoder_input, decoder_target,
    test_size=0.2, random_state=42, shuffle=True)

history = model.fit([X_question_train, X_decoder_train], y_train, validation_data=([X_question_val, X_decoder_val],y_val), epochs=50, batch_size=32)

In [ ]:
# ============================================================
# 16. SAVE MODEL
# ============================================================

model.save("minibot_model.keras")
# Save question tokenizer
with open("question_tokenizer.json",   "w") as file:
    file.write(question_tokenizer.to_json())

# Save answer tokenizer
with open("answer_tokenizer.json", "w") as file:
    file.write(answer_tokenizer.to_json())

# Save configuration
config = {
    "max_question_length":max_question_length,
    "max_answer_length":max_answer_length
}

with open("minibot_config.json","w") as file:
    json.dump(config,file)

print("\nModel saved successfully!")

In [ ]:
# ============================================================
# 17. CREATE INDEX -> WORD
# ============================================================

index_to_word = {
    index: word
    for word, index
    in answer_tokenizer.word_index.items()
}

In [ ]:
# ============================================================
# 18. CHAT FUNCTION
# ============================================================

def chat(user_question):
    
    # ----------------------------------------
    # Clean question
    # ----------------------------------------

    user_question = clean_text(user_question)

    # ----------------------------------------
    # Find closest known question
    # ----------------------------------------

    matches = get_close_matches(user_question, questions, n=1, cutoff=0.6)

    if not matches:
        return (
            "I do not understand "
            "that question yet."
        )
    matched_question = matches[0]

    # ----------------------------------------
    # Convert question into numbers
    # ----------------------------------------

    question_sequence = question_tokenizer.texts_to_sequences([matched_question])
    question_sequence = pad_sequences(question_sequence,maxlen=max_question_length,padding="post")

    # ----------------------------------------
    # Start generating answer
    # ----------------------------------------

    start_token = answer_tokenizer.word_index["<start>"]
    generated_tokens = [start_token]
    generated_words = []

    # ----------------------------------------
    # Generate one word at a time
    # ----------------------------------------

    for position in range(max_answer_length - 1):

        decoder_sequence = pad_sequences([generated_tokens],maxlen=max_answer_length - 1,padding="post")
        prediction = model.predict([question_sequence,decoder_sequence],verbose=0)
        predicted_id = np.argmax(prediction[0,position])
        predicted_word = (index_to_word.get(predicted_id))

        # Stop when answer is finished
        if (predicted_word is None or predicted_word == "<end>"):
            break

        # Ignore special tokens
        if predicted_word not in ["<start>", "<unk>"]:
            generated_words.append(predicted_word)

        generated_tokens.append(predicted_id)

    # ----------------------------------------
    # Convert words into sentence
    # ----------------------------------------

    if not generated_words:
        return (
            "I do not understand "
            "that question yet."
        )

    return " ".join(generated_words)

In [ ]:
# ============================================================
# 19. START CHATBOT
# ============================================================

while True:
    user_message = input("\nYou: ")

    if clean_text(user_message) in ["exit","quit"]:
        print("Bot: Goodbye!")
        break

    answer = chat(user_message)
    
    print("Bot:", answer)